# Tema 03 — Detectores de Bordas por Gradiente
**Operadores de Prewitt, Sobel e Scharr**

- **Disciplina:** Computação Gráfica
- **Grupo:** Eduardo Ribeiro · Vitor Paloco · Diego Vinicius
- **Objetivo:** Demonstrar na prática o cálculo do gradiente de imagens digitais em Python, comparando os operadores Prewitt, Sobel e Scharr e analisando o impacto dos principais parâmetros (tamanho do kernel, limiar, ruído/pré-suavização e norma da magnitude).


## 1. Bibliotecas e Carregamento da Imagem Base


In [ ]:
import cv2                         # OpenCV: filtros de convolução (Sobel, Scharr, GaussianBlur)
import numpy as np                 # NumPy: operações matriciais, magnitude e percentis
import matplotlib.pyplot as plt    # Matplotlib: exibição das imagens e comparações visuais
from skimage import data           # Dataset clássico de processamento de imagens

# Imagem clássica 'camera' em níveis de cinza (matriz 512x512 com valores uint8 de 0 a 255)
img = data.camera()

plt.figure(figsize=(4, 4))
plt.imshow(img, cmap="gray")
plt.title(f"Imagem original ({img.shape[0]}×{img.shape[1]} px)")
plt.axis("off")
plt.show()


## 2. Pipeline Completo e Passos Intermediários (Sobel 3×3)
Cada etapa transforma a imagem até isolar os contornos:
1. **`Gx` (derivada horizontal):** compara esquerda e direita → detecta **bordas verticais**.
2. **`Gy` (derivada vertical):** compara cima e baixo → detecta **bordas horizontais**.
3. **Magnitude `|∇f| = √(Gx² + Gy²)`:** combina os dois eixos em uma única força de borda.
4. **Binarização (`|∇f| >= T`):** pixels acima do limiar acendem em branco (borda); abaixo ficam pretos (fundo).


In [ ]:
# Derivadas parciais com Sobel 3x3:
# CV_64F (ponto flutuante 64 bits) evita perder valores negativos e suporta magnitudes > 255
gx = cv2.Sobel(img, cv2.CV_64F, dx=1, dy=0, ksize=3)  # variação em X -> bordas verticais
gy = cv2.Sobel(img, cv2.CV_64F, dx=0, dy=1, ksize=3)  # variação em Y -> bordas horizontais

# Magnitude Euclidiana (L2): combina Gx e Gy em todas as orientações
magnitude = np.sqrt(gx**2 + gy**2)

# Limiarização pelo percentil 90: seleciona os 10% de pixels com maior gradiente
limiar = np.percentile(magnitude, 90)
bordas = magnitude >= limiar  # True (branco) = borda | False (preto) = fundo

# Exibição sequencial de todas as etapas do pipeline
fig, axes = plt.subplots(1, 5, figsize=(16, 3.5))
etapas = [
    (img,         "1. Original"),
    (np.abs(gx),  "2. |Gx| (verticais)"),
    (np.abs(gy),  "3. |Gy| (horizontais)"),
    (magnitude,   "4. Magnitude |∇f|"),
    (bordas,      f"5. Binarizada (T={limiar:.0f})"),
]
for ax, (imagem, titulo) in zip(axes, etapas):
    ax.imshow(imagem, cmap="gray")
    ax.set_title(titulo)
    ax.axis("off")
plt.tight_layout()
plt.show()


## 3. Comparação entre Prewitt, Sobel e Scharr (3×3)
Os três operadores aplicam a mesma derivada `[-1, 0, +1]`, variando apenas os pesos de suavização perpendicular:
- **Prewitt:** pesos `[1, 1, 1]` (média simples, aplicado via `cv2.filter2D`)
- **Sobel:** pesos `[1, 2, 1]` (aproximação binomial/gaussiana — padrão da indústria)
- **Scharr:** pesos `[3, 10, 3]` (otimizado para menor erro angular em 3×3)


In [ ]:
# --- 1) Prewitt: máscara 3x3 explícita (soma dos pesos positivos = 3) ---
kernel_prewitt_x = np.array([[-1, 0, 1],
                             [-1, 0, 1],
                             [-1, 0, 1]], dtype=np.float64)
kernel_prewitt_y = kernel_prewitt_x.T  # transposta mede variação na vertical

gx_p = cv2.filter2D(img.astype(np.float64), -1, kernel_prewitt_x)
gy_p = cv2.filter2D(img.astype(np.float64), -1, kernel_prewitt_y)
mag_prewitt = np.sqrt(gx_p**2 + gy_p**2)

# --- 2) Sobel 3x3 (soma dos pesos positivos = 4) ---
gx_so = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=3)
gy_so = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=3)
mag_sobel = np.sqrt(gx_so**2 + gy_so**2)

# --- 3) Scharr 3x3 (soma dos pesos positivos = 16 -> escala numérica maior) ---
gx_sc = cv2.Scharr(img, cv2.CV_64F, 1, 0)
gy_sc = cv2.Scharr(img, cv2.CV_64F, 0, 1)
mag_scharr = np.sqrt(gx_sc**2 + gy_sc**2)

# A olho nu os resultados no percentil 90 são muito próximos; a escala (máx e T) é que muda
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (nome, mag) in zip(axes, [("Prewitt", mag_prewitt), ("Sobel", mag_sobel), ("Scharr", mag_scharr)]):
    t = np.percentile(mag, 90)
    ax.imshow(mag >= t, cmap="gray")
    ax.set_title(f"{nome} | máx={mag.max():.0f} | T(p90)={t:.0f}")
    ax.axis("off")
plt.tight_layout()
plt.show()


## 4. Função Unificada para Análise de Parâmetros
Centraliza o pipeline completo (ruído sintético opcional → pré-suavização Gaussiana → cálculo de `Gx`/`Gy` → magnitude `L2`/`L1` → binarização) para testar cada parâmetro isoladamente.


In [ ]:
def detectar_bordas(operador="sobel", ksize=3, limiar=None, ruido=False, blur=0, modo="L2"):
    """Executa o detector de bordas por gradiente com parâmetros configuráveis."""
    # 1) Opcional: adiciona ruído Gaussiano (sigma=20) com semente fixa para reprodutibilidade
    entrada = img.copy()
    if ruido:
        np.random.seed(42)
        entrada = np.clip(img + np.random.normal(0, 20, img.shape), 0, 255).astype(np.uint8)

    # 2) Opcional: pré-suavização Gaussiana para atenuar ruído antes de derivar
    if blur > 0:
        entrada = cv2.GaussianBlur(entrada, (blur, blur), 0)

    # 3) Cálculo das derivadas Gx e Gy conforme o operador escolhido
    if operador == "prewitt":
        gx = cv2.filter2D(entrada.astype(np.float64), -1, kernel_prewitt_x)
        gy = cv2.filter2D(entrada.astype(np.float64), -1, kernel_prewitt_y)
    elif operador == "scharr":
        gx = cv2.Scharr(entrada, cv2.CV_64F, 1, 0)
        gy = cv2.Scharr(entrada, cv2.CV_64F, 0, 1)
    else:
        gx = cv2.Sobel(entrada, cv2.CV_64F, 1, 0, ksize=ksize)
        gy = cv2.Sobel(entrada, cv2.CV_64F, 0, 1, ksize=ksize)

    # 4) Magnitude: L2 (exata/isotrópica) ou L1 (soma dos módulos, mais rápida)
    mag = np.sqrt(gx**2 + gy**2) if modo == "L2" else (np.abs(gx) + np.abs(gy))

    # 5) Binarização: usa limiar fixo se informado, ou percentil 90 automático
    t = np.percentile(mag, 90) if limiar is None else limiar
    binaria = mag >= t
    return mag, binaria, t


### Experimento 1 — Diferença de Escala entre Operadores (Limiar Fixo `T = 150`)
Mantendo o mesmo limiar absoluto, **Prewitt** marca menos bordas e **Scharr** marca muito mais porque a soma dos pesos positivos varia (`3`, `4` e `16`). Ao trocar de operador, é necessário reajustar o limiar.


In [ ]:
# Mesmo limiar absoluto (T=150) aplicado aos 3 operadores para evidenciar a diferença de escala
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, op in zip(axes, ["prewitt", "sobel", "scharr"]):
    mag, binaria, t = detectar_bordas(operador=op, limiar=150)
    ax.imshow(binaria, cmap="gray")
    ax.set_title(f"{op.upper()} | máx={mag.max():.0f} | T fixo={t}")
    ax.axis("off")
plt.tight_layout()
plt.show()


### Experimento 2 — Impacto do Tamanho do Kernel (`ksize`: 3×3, 5×5 e 7×7)
Kernels maiores aplicam mais suavização: as bordas ficam mais grossas, detalhes finos desaparecem e a magnitude máxima cresce rapidamente devido aos pesos binomiais.


In [ ]:
# Comparação de Sobel 3x3, 5x5 e 7x7 com limiar proporcional (percentil 90)
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, k in zip(axes, [3, 5, 7]):
    mag, binaria, t = detectar_bordas(operador="sobel", ksize=k, limiar=None)
    ax.imshow(binaria, cmap="gray")
    ax.set_title(f"Sobel {k}×{k} | máx={mag.max():,.0f} | T(p90)={t:,.0f}")
    ax.axis("off")
plt.tight_layout()
plt.show()


### Experimento 3 — Sensibilidade ao Ruído e Pré-Suavização Gaussiana
A derivada amplifica variações bruscas causadas por ruído (falsos positivos). Aplicar `GaussianBlur` antes do gradiente elimina o ruído; porém, um blur excessivo (`7×7`) começa a alargar e deslocar as bordas.


In [ ]:
# Imagem ruidosa sem filtro vs pré-suavização Gaussiana 3x3, 5x5 e 7x7 (Sobel 3x3, T=120)
fig, axes = plt.subplots(1, 4, figsize=(15, 3.8))
configuracoes = [
    ("Sem blur (ruído amplificado)", 0),
    ("Gaussiano 3×3",                3),
    ("Gaussiano 5×5 (equilíbrio)",   5),
    ("Gaussiano 7×7 (bordas largas)", 7),
]
for ax, (titulo, b) in zip(axes, configuracoes):
    mag, binaria, t = detectar_bordas(operador="sobel", ruido=True, blur=b, limiar=120)
    ax.imshow(binaria, cmap="gray")
    ax.set_title(titulo)
    ax.axis("off")
plt.tight_layout()
plt.show()


### Experimento 4 — Magnitude `L2` (Euclidiana) vs `L1` (Soma dos Módulos)
- **L2 (`√(Gx² + Gy²)`):** isotrópica (responde igualmente a todas as orientações).
- **L1 (`|Gx| + |Gy|`):** evita a raiz quadrada (menor custo em hardware simples), mas superestima bordas diagonais em até **41% (`√2`)**.


In [ ]:
# Comparação direta entre norma L2 e norma L1 sob o mesmo limiar absoluto (T=150)
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, modo in zip(axes, ["L2", "L1"]):
    mag, binaria, t = detectar_bordas(operador="sobel", limiar=150, modo=modo)
    ax.imshow(binaria, cmap="gray")
    ax.set_title(f"Magnitude {modo} | máx={mag.max():.0f} | T={t}")
    ax.axis("off")
plt.tight_layout()
plt.show()
